In [ ]:
# ============================================================
# FLUX.2 Klein (4B) — LoRA / AdaLoRA / IA3 Fine-Tuning Benchmark
# ГИПЕРПАРАМЕТРЫ: MAX_STEPS=800, LR=5e-4, GRAD_ACCUM=2
# ============================================================
import os, gc, re, warnings, logging, traceback, random, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, numpy as np, pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import clip
from tqdm.auto import tqdm

from diffusers import (
    Flux2KleinPipeline,
    Flux2Transformer2DModel,
    AutoencoderKLFlux2,
    FlowMatchEulerDiscreteScheduler,
)
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import (
    LoraConfig, AdaLoraConfig, IA3Config,
    get_peft_model, PeftModel,
)
from huggingface_hub import login
from google.colab import drive, userdata

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

# ============================================================
# 0. Seed и TF32
# ============================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ============================================================
# 1. Проверка VRAM
# ============================================================
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"VRAM: free={free/1e9:.2f} GB / total={total/1e9:.2f} GB")
if free / 1e9 < 12.0:
    raise SystemExit("Мало VRAM. Runtime → Restart runtime.")

# ============================================================
# 2. HF + Drive
# ============================================================
try:
    login(token=userdata.get("HF_TOKEN")); print("✓ Логин в HF")
except Exception:
    print("⚠ HF_TOKEN не найден")
drive.mount('/content/drive')

# ============================================================
# 3. Гиперпараметры 
# ============================================================
BASE_DIR    = "/content/drive/MyDrive/Colab Notebooks/LoraDataset"
OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/flux_results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
metadata_path = os.path.join(BASE_DIR, "metadata.csv")

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"

IMAGE_SIZE  = 192
LATENT_SIZE = IMAGE_SIZE // 8          # 24
MAX_STEPS   = 800                      # 800
LR          = 5e-4                     # 5e-4
GRAD_ACCUM  = 2                        # 2
LIMIT_N     = 100
MAX_TOKENS  = 128
BATCH_SIZE  = 1

print(f"\n{'='*60}")
print(f"  ГИПЕРПАРАМЕТРЫ ОБУЧЕНИЯ")
print(f"{'='*60}")
print(f"  IMAGE_SIZE  = {IMAGE_SIZE}")
print(f"  MAX_STEPS   = {MAX_STEPS}")
print(f"  LR          = {LR}")
print(f"  GRAD_ACCUM  = {GRAD_ACCUM}")
print(f"  Обновлений оптимизатора: {MAX_STEPS // GRAD_ACCUM}")
print(f"{'='*60}\n")

prompts = [
    "qwerty style, A work by an unknown artist from the 18th century. Oval, depicting a bouquet of flowers on a dark background",
    "qwerty style, A work by an unknown artist from the 18th century. Square, depicting a landscape with a tower on a rocky riverbank",
    "qwerty style, A work by an unknown artist from the 18th century. Rectangular, depicting a landscape with a waterfall in the foreground, a hut in the background",
]

# ============================================================
# 4. Tokenizer и Scheduler
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# ============================================================
# 5. Датасет
# ============================================================
df = pd.read_csv(metadata_path, encoding="utf-8")

def normalize_path(x):
    x = str(x).strip().strip('"').strip("'").replace("\\", "/")
    return os.path.normpath(x) if os.path.isabs(x) else os.path.normpath(os.path.join(BASE_DIR, x))

df["file_name"] = df["file_name"].apply(normalize_path)
df = df.head(LIMIT_N).reset_index(drop=True)
df = df[df["file_name"].apply(os.path.exists)].reset_index(drop=True)
print(f"Используем {len(df)} изображений")

class DS(Dataset):
    def __init__(self, df, tok, size=192):
        self.df, self.tok = df, tok
        self.t = transforms.Compose([
            transforms.Resize(size), transforms.CenterCrop(size),
            transforms.ToTensor(), transforms.Normalize([0.5], [0.5]),
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        px = self.t(Image.open(r["file_name"]).convert("RGB"))
        ids = self.tok(r["caption"], padding="max_length", max_length=MAX_TOKENS,
                       truncation=True, return_tensors="pt")["input_ids"].squeeze(0)
        return px, ids

ds = DS(df, tokenizer, size=IMAGE_SIZE)

# ============================================================
# 6. Patchify helper
# ============================================================
def patchify_latents(lat):
    """[B, 32, H, W] → [B, (H/2)*(W/2), 128]"""
    B, C, H, W = lat.shape
    lat = lat.view(B, C, H // 2, 2, W // 2, 2).permute(0, 2, 4, 1, 3, 5)
    return lat.reshape(B, (H // 2) * (W // 2), C * 4)

# ============================================================
# 7. Предвычисление латентов с BN-нормализацией
# ============================================================
print("1. Предвычисление латентов (с BN-нормализацией)...")
vae = AutoencoderKLFlux2.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.bfloat16)
vae.to("cuda")

BN_EPS  = getattr(vae.config, "batch_norm_eps", 1e-4)
BN_MEAN = vae.bn.running_mean.clone()
BN_STD  = torch.sqrt(vae.bn.running_var + BN_EPS)

print(f"   BN eps={BN_EPS}, mean={BN_MEAN.mean().item():.4f}, std={BN_STD.mean().item():.4f}")

BN_MEAN_V = BN_MEAN.view(1, 1, -1)
BN_STD_V  = BN_STD.view(1, 1, -1)

all_latents = []
with torch.no_grad():
    for i in range(len(ds)):
        px, _ = ds[i]
        raw = vae.encode(px.unsqueeze(0).to("cuda", dtype=torch.bfloat16)).latent_dist.sample()
        packed = patchify_latents(raw)
        bn_m = BN_MEAN_V.to(packed.device, packed.dtype)
        bn_s = BN_STD_V.to(packed.device, packed.dtype)
        normed = (packed - bn_m) / bn_s
        all_latents.append(normed.to(torch.bfloat16).cpu())

vae.to("cpu"); del vae
gc.collect(); torch.cuda.empty_cache()
print(f"   латентов: {len(all_latents)}, форма: {all_latents[0].shape}")
print(f"   std после BN: {all_latents[0].float().std().item():.4f}")

# ============================================================
# 8. Предвычисление текстовых эмбеддингов
# ============================================================
print("\n2. Предвычисление эмбеддингов...")
text_encoder = AutoModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16),
    device_map="cuda", torch_dtype=torch.bfloat16,
).eval()

LAYERS = [9, 18, 27]
all_emb = []
with torch.no_grad():
    for i in range(len(ds)):
        _, ids = ds[i]
        out = text_encoder(ids.unsqueeze(0).to("cuda"), output_hidden_states=True)
        stacked = torch.cat([out.hidden_states[l] for l in LAYERS], dim=-1)
        all_emb.append(stacked.to(torch.bfloat16).cpu())

del text_encoder
gc.collect(); torch.cuda.empty_cache()
print(f"   эмбеддингов: {len(all_emb)}, форма: {all_emb[0].shape}")

# ============================================================
# 9. 4D RoPE ids
# ============================================================
def make_img_ids(latent_H, latent_W, device):
    """4D RoPE: [1, S, 4] = [t, row, col, extra]"""
    h, w = latent_H // 2, latent_W // 2
    t = torch.zeros(1, h * w, 1, device=device, dtype=torch.float32)
    row = torch.arange(h, device=device).repeat_interleave(w).unsqueeze(0).unsqueeze(-1).float()
    col = torch.arange(w, device=device).repeat(h).unsqueeze(0).unsqueeze(-1).float()
    extra = torch.zeros(1, h * w, 1, device=device, dtype=torch.float32)
    return torch.cat([t, row, col, extra], dim=-1)

def make_txt_ids(ehs):
    return torch.zeros(1, ehs.shape[1], 4, device=ehs.device, dtype=torch.float32)

print(f"\n latent: {all_latents[0].shape}, emb: {all_emb[0].shape}")
print(f" LATENT_SIZE={LATENT_SIZE}, патчей: {(LATENT_SIZE//2)**2}")
assert all_latents[0].shape[-1] == 128
assert all_latents[0].shape[1] == (LATENT_SIZE // 2) ** 2

# ============================================================
# 10. Конфигурации PEFT
# ============================================================
TARGET_MODULES = ["to_q", "to_k", "to_v"]
OPT_STEPS = MAX_STEPS // GRAD_ACCUM

configs = {
    "lora": LoraConfig(
        r=8, lora_alpha=8, target_modules=TARGET_MODULES,
        lora_dropout=0.1, bias="none",
    ),
    "adalora": AdaLoraConfig(
        r=8, lora_alpha=8, target_modules=TARGET_MODULES,
        lora_dropout=0.1, bias="none",
        total_step=OPT_STEPS, init_r=12, target_r=4,
        beta1=0.85, beta2=0.85, orth_reg_weight=0.5,
    ),
    "ia3": IA3Config(
        target_modules=TARGET_MODULES, feedforward_modules=[],
    ),
}
METHODS = ["lora", "adalora", "ia3"]

# ============================================================
# 10a. Быстрый тест: проверка timestep
# ============================================================
print("\n0. Проверка timestep (sigma vs raw)...")
_tf_probe = Flux2Transformer2DModel.from_pretrained(
    MODEL_ID, subfolder="transformer", torch_dtype=torch.bfloat16,
).to("cuda").eval()

_packed = all_latents[0].to("cuda")
_ehs    = all_emb[0].to("cuda")
_img_ids = make_img_ids(LATENT_SIZE, LATENT_SIZE, "cuda")
_txt_ids = make_txt_ids(_ehs)

_u  = torch.sigmoid(torch.randn(1, device="cuda"))
_ts = _u * 1000.0
_t  = _u.to(torch.bfloat16)[:, None, None]
_noise = torch.randn_like(_packed)
_noisy = (1 - _t) * _packed + _t * _noise
_target = _noise - _packed

with torch.no_grad():
    _pred_a = _tf_probe(
        hidden_states=_noisy, encoder_hidden_states=_ehs,
        timestep=_u.to(torch.bfloat16),
        img_ids=_img_ids, txt_ids=_txt_ids,
        guidance=None, return_dict=False,
    )[0]
    _loss_a = torch.nn.functional.mse_loss(_pred_a.float(), _target.float()).item()

    _pred_b = _tf_probe(
        hidden_states=_noisy, encoder_hidden_states=_ehs,
        timestep=_ts.to(torch.bfloat16),
        img_ids=_img_ids, txt_ids=_txt_ids,
        guidance=None, return_dict=False,
    )[0]
    _loss_b = torch.nn.functional.mse_loss(_pred_b.float(), _target.float()).item()

print(f"   latent std = {_packed.float().std().item():.4f}")
print(f"   timestep sigma [0,1]   → loss = {_loss_a:.4f}")
print(f"   timestep raw  [0,1000] → loss = {_loss_b:.4f}")
TIMESTEP_SIGMA = _loss_a < _loss_b
print(f"   → Используем sigma space: {TIMESTEP_SIGMA}")

del _tf_probe
gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 11. Обучение
# ============================================================
def train_method(method):
    out_dir = os.path.join(OUTPUT_ROOT, f"{method}_results")
    os.makedirs(out_dir, exist_ok=True)
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n{'='*60}\n {method.upper()}  ({MAX_STEPS} steps, LR={LR}, GA={GRAD_ACCUM})\n{'='*60}")

    train_tf = Flux2Transformer2DModel.from_pretrained(
        MODEL_ID, subfolder="transformer", torch_dtype=torch.bfloat16,
    )
    train_tf.to("cuda")
    train_tf.enable_gradient_checkpointing()
    train_tf.config.use_cache = False

    model = get_peft_model(train_tf, configs[method])
    model.print_trainable_parameters()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    model.train()
    step = 0
    pbar = tqdm(total=MAX_STEPS, desc=f"{method}")
    t0 = time.time()
    loss_history = []

    try:
        indices = list(range(len(all_latents)))
        while step < MAX_STEPS:
            random.shuffle(indices)
            for idx in indices:
                if step >= MAX_STEPS:
                    break

                packed = all_latents[idx].to("cuda")
                ehs    = all_emb[idx].to("cuda")
                img_ids = make_img_ids(LATENT_SIZE, LATENT_SIZE, "cuda")
                txt_ids = make_txt_ids(ehs)

                u = torch.sigmoid(torch.randn((packed.shape[0],), device="cuda", dtype=torch.float32))
                ts = u * 1000.0
                t  = u.to(torch.bfloat16)[:, None, None]
                noise = torch.randn_like(packed)
                noisy = (1 - t) * packed + t * noise
                target = noise - packed

                ts_model = u if TIMESTEP_SIGMA else ts
                pred = model(
                    hidden_states=noisy,
                    encoder_hidden_states=ehs,
                    timestep=ts_model.to(torch.bfloat16),
                    img_ids=img_ids,
                    txt_ids=txt_ids,
                    guidance=None,
                    return_dict=False,
                )[0]

                loss = torch.nn.functional.mse_loss(pred.float(), target.float()) / GRAD_ACCUM
                loss.backward()

                if (step + 1) % GRAD_ACCUM == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                loss_val = loss.item() * GRAD_ACCUM
                loss_history.append(loss_val)

                if step % 40 == 0:
                    elapsed = time.time() - t0
                    eta = elapsed / max(step + 1, 1) * (MAX_STEPS - step)
                    recent = np.mean(loss_history[-40:]) if len(loss_history) >= 40 else loss_val
                    print(f"   step {step:>4}, loss: {loss_val:.4f}, "
                          f"avg(last40): {recent:.4f}, ETA: {eta/60:.1f} min")

                step += 1
                pbar.update(1)

        model.save_pretrained(out_dir)
        print(f"✓ {method} сохранён: {out_dir}")
        print(f"   Финальный avg(last100): {np.mean(loss_history[-100:]):.4f}")
        ok = True
    except Exception as e:
        traceback.print_exc()
        print(f"✗ {method} FAILED: {e}")
        ok = False
    finally:
        pbar.close()

    del model, optimizer, train_tf
    gc.collect(); torch.cuda.empty_cache()
    return ok

trained = []
for m in METHODS:
    if train_method(m):
        trained.append(m)
print(f"\n✓ Обучены: {trained}")

# ============================================================
# 12. Генерация
# ============================================================
print("\n" + "=" * 60 + "\nГЕНЕРАЦИЯ\n" + "=" * 60)

if trained:
    vae_gen = AutoencoderKLFlux2.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.bfloat16)
    vae_gen.to("cuda")

    text_encoder_gen = AutoModel.from_pretrained(
        MODEL_ID, subfolder="text_encoder",
        quantization_config=BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16),
        device_map="cuda", torch_dtype=torch.bfloat16,
    ).eval()

    for method in trained:
        adapter_dir = os.path.join(OUTPUT_ROOT, f"{method}_results")
        print(f"\n=== {method} ===")
        try:
            tf = Flux2Transformer2DModel.from_pretrained(
                MODEL_ID, subfolder="transformer", torch_dtype=torch.bfloat16,
            )
            tf = PeftModel.from_pretrained(tf, adapter_dir)
            try:
                tf = tf.merge_and_unload()
                print(f"   {method}: adapter merged")
            except Exception as merge_err:
                print(f"   {method}: merge_and_unload недоступен ({merge_err})")
            tf.to("cuda")

            pipe = Flux2KleinPipeline(
                transformer=tf, vae=vae_gen, text_encoder=text_encoder_gen,
                tokenizer=tokenizer, scheduler=scheduler,
            )
            pipe.enable_attention_slicing()

            for i, p in enumerate(prompts):
                try:
                    img = pipe(prompt=p, height=512, width=512, guidance_scale=1.0,
                               num_inference_steps=4,
                               generator=torch.Generator(device="cuda").manual_seed(SEED)).images[0]
                except torch.cuda.OutOfMemoryError:
                    print(f"   {method} {i} OOM → 384x384")
                    torch.cuda.empty_cache()
                    img = pipe(prompt=p, height=384, width=384, guidance_scale=1.0,
                               num_inference_steps=4,
                               generator=torch.Generator(device="cuda").manual_seed(SEED)).images[0]
                img.save(os.path.join(OUTPUT_ROOT, f"{method}_result_{i}.png"))
                print(f"   saved {method} {i}")

            del pipe, tf
            gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f"   ✗ {method}: {e}")
            traceback.print_exc()
            gc.collect(); torch.cuda.empty_cache()

    del vae_gen, text_encoder_gen
    gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 13. CLIP + сетка
# ============================================================
print("\n" + "=" * 60 + "\nCLIP\n" + "=" * 60)

device = "cuda"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

method_files = {}
for fname in os.listdir(OUTPUT_ROOT):
    m = re.match(r"(.+)_result_(\d+)\.png", fname)
    if m:
        method_files.setdefault(m.group(1), {})[int(m.group(2))] = os.path.join(OUTPUT_ROOT, fname)

methods_present = sorted(method_files.keys())
print(f"Найдены методы: {methods_present}")

if methods_present:
    num_prompts = max(len(v) for v in method_files.values())
    results = {}
    for method in methods_present:
        scores = []
        for i in range(num_prompts):
            fpath = method_files[method].get(i)
            if not fpath or not os.path.exists(fpath):
                scores.append(None); continue
            try:
                image = Image.open(fpath).convert("RGB")
                img_t = preprocess(image).unsqueeze(0).to(device)
                text = prompts[i] if i < len(prompts) else "qwerty style"
                txt_t = clip.tokenize([text]).to(device)
                with torch.no_grad():
                    fi = clip_model.encode_image(img_t); fi /= fi.norm(dim=1, keepdim=True)
                    ft = clip_model.encode_text(txt_t); ft /= ft.norm(dim=1, keepdim=True)
                    scores.append((fi @ ft.T).item())
            except Exception:
                scores.append(None)
        results[method] = scores

    print("\nCLIP Scores:")
    print("Method    | " + " | ".join([f"P{i}" for i in range(num_prompts)]) + " | Avg")
    for method in methods_present:
        s = results[method]
        valid = [x for x in s if x is not None]
        avg = np.mean(valid) if valid else 0
        sstr = " | ".join([f"{x:.4f}" if x is not None else "N/A" for x in s])
        print(f"{method:<9} | {sstr} | {avg:.4f}")

    rows, cols = len(methods_present), num_prompts
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    if rows == 1: axes = [axes]
    for r, method in enumerate(methods_present):
        for c in range(cols):
            fpath = method_files[method].get(c)
            if fpath and os.path.exists(fpath):
                axes[r][c].imshow(mpimg.imread(fpath))
                score = results[method][c] if results[method][c] is not None else 0.0
                axes[r][c].set_title(f"{method}\n{score:.3f}", fontsize=10)
            else:
                axes[r][c].text(0.5, 0.5, "N/A", ha="center", va="center")
            axes[r][c].axis("off")
    plt.tight_layout()
    grid_path = os.path.join(OUTPUT_ROOT, "comparison_grid.png")
    plt.savefig(grid_path, dpi=150); plt.show()
    print(f"\n✓ Сетка: {grid_path}")

print(f"\n✓ DONE: {OUTPUT_ROOT}")

Output hidden; open in https://colab.research.google.com to view.

In [1]:
!pip uninstall -y diffusers peft transformers accelerate bitsandbytes xformers triton -q
!pip install git+https://github.com/huggingface/diffusers.git -q
!pip install git+https://github.com/huggingface/peft.git -q
!pip install transformers accelerate datasets bitsandbytes sentencepiece -q
!pip install --no-deps --upgrade "torchao>=0.17.0" -q
!pip install git+https://github.com/openai/CLIP.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
